Bronze Ingestion Notebook

In [0]:
import json
import os
import time
from datetime import datetime, timezone
import requests

# 1. Fetch Credentials and Configuration
API_KEY = dbutils.secrets.get(scope="youtube_scope", key="api_key")
BASE_URL = "https://www.googleapis.com/youtube/v3"
VOLUME_LANDING_PATH = "/Volumes/youtube_lakehouse/bronze/raw_landing/videos"

# Seed channels for data engineering/tech content
# UCxladMszXan-jfgzyeQNwvw -> Databricks
# UCVpW4e68eA2M0Y5V_2Z1W1w -> Seattle Data Guy
# UCu7v6_bU-gE5f6rY0qS_Vqg -> Luke Barousse
TARGET_CHANNEL_IDS = [
    'UCf1XIplYiqNv9baGsFHwPPQ', 
    'UC3vHW2h22WE-pNi5WJtRIjg',
    'UCWJPKXhkcMGXafdtqGx1mEw',
    'UCtm8rtofLSnaIBi3noB0INg',
    'UCa9qP2KdOiWsBzWe6YD2ILw'
]

def make_api_request(endpoint: str, params: dict, retries: int = 3, backoff: int = 2) -> dict:
    """Wrapper with exponential backoff for quota resilience."""
    params["key"] = API_KEY
    url = f"{BASE_URL}/{endpoint}"
    
    for attempt in range(retries):
        response = requests.get(url, params=params)
        if response.status_code == 200:
            return response.json()
        elif response.status_code in [429, 500, 503]:
            time.sleep(backoff ** attempt)
        else:
            raise RuntimeError(f"API Call failed [{response.status_code}]: {response.text}")
            
    raise RuntimeError(f"Failed to fetch data from {endpoint} after {retries} attempts.")

def get_uploads_playlist_id(channel_id: str) -> str:
    """Fetches channel uploads playlist (Cost: 1 quota unit)."""
    data = make_api_request("channels", {
        "part": "contentDetails",
        "id": channel_id
    })
    items = data.get("items", [])
    if not items:
        return None
    return items[0]["contentDetails"]["relatedPlaylists"]["uploads"]

def get_recent_video_ids(playlist_id: str, max_results: int = 50) -> list:
    """Fetches video IDs from upload playlist (Cost: 1 quota unit per 50 items)."""
    data = make_api_request("playlistItems", {
        "part": "contentDetails",
        "playlistId": playlist_id,
        "maxResults": min(max_results, 50)
    })
    return [item["contentDetails"]["videoId"] for item in data.get("items", [])]

def get_video_payloads(video_ids: list) -> list:
    """Fetches snippet, contentDetails, and statistics in batches of 50 (Cost: 1 quota unit)."""
    if not video_ids:
        return []
    
    batched_results = []
    # Batch in slices of 50 IDs to maximize API efficiency
    for i in range(0, len(video_ids), 50):
        chunk = video_ids[i:i + 50]
        data = make_api_request("videos", {
            "part": "snippet,contentDetails,statistics",
            "id": ",".join(chunk)
        })
        batched_results.extend(data.get("items", []))
        
    return batched_results

# Execute Extraction Run
print(f"Starting YouTube API Extraction for {len(TARGET_CHANNEL_IDS)} channels...")
all_extracted_videos = []

for channel_id in TARGET_CHANNEL_IDS:
    uploads_id = get_uploads_playlist_id(channel_id)
    if uploads_id:
        video_ids = get_recent_video_ids(uploads_id, max_results=50)
        videos = get_video_payloads(video_ids)
        all_extracted_videos.extend(videos)
        print(f"Channel {channel_id}: Fetched {len(videos)} video records.")

print(f"Total video payloads extracted: {len(all_extracted_videos)}")

In [0]:
# Create partitioned directory path inside UC Volume
now = datetime.now(timezone.utc)
year_str = now.strftime("%Y")
month_str = now.strftime("%m")
batch_timestamp = now.strftime("%Y%m%d_%H%M%S")

target_volume_dir = f"{VOLUME_LANDING_PATH}/year={year_str}/month={month_str}"
os.makedirs(f"/Volumes{target_volume_dir.replace('/Volumes', '')}", exist_ok=True)

# Wrap payload with ingestion metadata
envelope = {
    "_source_batch_id": batch_timestamp,
    "_extracted_at_utc": now.isoformat(),
    "record_count": len(all_extracted_videos),
    "records": all_extracted_videos
}

file_target = f"{target_volume_dir}/youtube_batch_{batch_timestamp}.json"

with open(file_target, "w", encoding="utf-8") as f:
    json.dump(envelope, f, ensure_ascii=False)

print(f"Successfully landed batch to: {file_target}")

In [0]:
display(dbutils.fs.ls(VOLUME_LANDING_PATH))

In [0]:
from pyspark.sql.functions import current_timestamp, col

# Storage Paths
volume_source_path = "/Volumes/youtube_lakehouse/bronze/raw_landing/videos"
checkpoint_path = "/Volumes/youtube_lakehouse/bronze/raw_landing/_checkpoints/bronze_videos"
target_table = "youtube_lakehouse.bronze.raw_videos"

# 1. Stream Ingest using Auto Loader (cloudFiles)
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .option("multiline", "true")
    .load(volume_source_path)
    .withColumn("_ingest_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

# 2. Write to Bronze Delta Table using AvailableNow trigger (micro-batch)
query = (
    df_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()
print(f"Bronze table `{target_table}` successfully updated.")

In [0]:
%sql
SELECT 
    _source_batch_id,
    _ingest_timestamp,
    record_count,
    _source_file,
    _rescued_data
FROM youtube_lakehouse.bronze.raw_videos
LIMIT 10;

In [0]:
%sql
SELECT 
    _source_batch_id,
    _ingest_timestamp,
    EXPLODE(records) AS video_payload
FROM youtube_lakehouse.bronze.raw_videos
LIMIT 10;

In [0]:
%sql
SELECT 
    _source_batch_id,
    record_count,
    _ingest_timestamp,
    _source_file,
    SIZE(records) AS actual_records_in_array,
    _rescued_data
FROM youtube_lakehouse.bronze.raw_videos;

In [0]:
%sql
SELECT 
    v.id AS raw_video_id,
    v.snippet.channelTitle AS channel_title,
    v.snippet.title AS video_title,
    v.snippet.publishedAt AS published_at,
    v.statistics.viewCount AS raw_view_count,
    v.snippet.tags AS tags_array
FROM youtube_lakehouse.bronze.raw_videos
LATERAL VIEW EXPLODE(records) AS v
LIMIT 500;

In [0]:
%sql
select 
       v.contentdetails.*,
       v.statistics.*,
       v.snippet.localized.*
FROM youtube_lakehouse.bronze.raw_videos
LATERAL VIEW EXPLODE(records) AS v
LIMIT 5;